# 02 – Core database setup (UDA-Hub)
Creates `data/core/udahub.db` with tables **accounts, users, tickets, ticket_metadata, ticket_messages, knowledge** (+ `customer_memory` for long-term memory), loads the CultPass knowledge base from `data/core/cultpass_articles.jsonl` (**18 articles**: the 4 provided + 14 new, 7 categories) and mirrors the CultPass users.

In [1]:
from data.setup_db import setup_core_db
setup_core_db()

{'accounts': 1,
 'users': 8,
 'tickets': 0,
 'ticket_metadata': 0,
 'ticket_messages': 0,
 'knowledge': 18,
 'customer_memory': 0}

In [2]:
from sqlalchemy import select, func
from agentic.db import udahub_session
from data.models.udahub import Knowledge
with udahub_session() as db:
    rows = db.execute(select(Knowledge.category, func.count()).group_by(Knowledge.category)).all()
    titles = db.execute(select(Knowledge.article_id, Knowledge.category, Knowledge.title)).all()
print(dict(rows)); print(len(titles), 'articles')
for t in titles: print(*t, sep=' | ')

{'account': 3, 'billing': 5, 'experiences': 1, 'general': 1, 'reservations': 3, 'subscription': 2, 'technical': 3}
18 articles
kb-001 | reservations | How to reserve an experience
kb-002 | reservations | How to cancel a reservation
kb-003 | subscription | How to cancel your subscription
kb-004 | billing | Subscription plans and pricing
kb-005 | billing | Refund policy
kb-006 | billing | How to update your payment method
kb-007 | billing | Payment failed or card declined
kb-008 | billing | Download invoices and receipts
kb-009 | account | Reset your password or fix login problems
kb-010 | account | Update your email address or profile
kb-011 | account | Delete your account and personal data
kb-012 | subscription | Pause your subscription
kb-013 | technical | The app crashes, freezes or will not load
kb-014 | technical | QR code will not scan at the venue
kb-015 | technical | Not receiving notifications or emails
kb-016 | experiences | An experience was cancelled or rescheduled by the pa

Retrieval check (RAG, see `agentic/tools/kb_search.py`):

In [3]:
from agentic.tools.kb_search import search_knowledge
for q in ['how do I cancel my subscription', 'my QR code will not scan', 'weather in Paris']:
    r = search_knowledge('cultpass', q)['data']
    print(q, '->', [(x['title'], x['confidence']) for x in r['results'][:2]])

how do I cancel my subscription -> [('How to cancel your subscription', 0.5281), ('How to cancel a reservation', 0.1153)]
my QR code will not scan -> [('QR code will not scan at the venue', 0.477)]
weather in Paris -> []
